### 1. Setup

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data.loader import load_raw_data
from src.data.cleaner import clean_data
from src.data.features import engineer_features

df_raw = load_raw_data("../data/raw/AmesHousing.csv")
df_clean = clean_data(df_raw)
df = engineer_features(df_clean)

print(f"Shape: {df.shape}")
print(f"Target range: ${df['SalePrice'].min():,.0f} – ${df['SalePrice'].max():,.0f}")

INFO:src.data.loader:Loading dataset from ..\data\raw\AmesHousing.csv...
INFO:src.data.loader:Loaded 2,930 rows and 82 columns.
INFO:src.data.loader:Column validation passed. All required columns are present.
INFO:src.data.cleaner:Starting data cleaning pipeline ...
INFO:src.data.cleaner:Dropped 7 columns: ['Order', 'PID', 'Misc Feature', 'Misc Val', 'Pool QC', 'Fence', 'Alley']
INFO:src.data.cleaner:Filled numeric null in 'Lot Frontage' with median=68.00
INFO:src.data.cleaner:Filled numeric null in 'Garage Yr Blt' with median=1979.00
INFO:src.data.cleaner:Filled categorical null in 'Electrical' with mode='SBrkr'
INFO:src.data.cleaner:Removed 5 outlier rows. Remaining: 2,925
INFO:src.data.cleaner:Data cleaning completed. Final shape: (2925, 75)
INFO:src.data.features:Starting feature engineering...
INFO:src.data.features:Label-encoded 39 categorical columns.
INFO:src.data.features:Feature engineering complete. Shape: (2925, 88)


Shape: (2925, 88)
Target range: $12,789 – $625,000


### 2. Price Distribution

In [2]:
fig = px.histogram(
    df, x="SalePrice", nbins=60,
    title="Distribution of Sale Prices",
    labels={"SalePrice": "Sale Price (USD)"},
    color_discrete_sequence=["#2563eb"]
)
fig.update_layout(bargap=0.05)
fig.show()

### 3. Price By Neighborhood

In [3]:
neighborhood_stats = (
    df.groupby("Neighborhood")["SalePrice"]
    .median()
    .sort_values(ascending=False)
    .reset_index()
)

fig = px.bar(
    neighborhood_stats,
    x="Neighborhood", y="SalePrice",
    title="Median Sale Price by Neighborhood",
    labels={"SalePrice": "Median Price (USD)"},
    color="SalePrice",
    color_continuous_scale="Blues"
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

### 4. Quality vs Price

In [5]:
fig = px.box(
    df, x="Overall Qual", y="SalePrice",
    title="Sale Price by Overall Quality (1–10)",
    labels={"Overall Qual": "Quality Score", "SalePrice": "Sale Price"},
    color="Overall Qual",
    color_discrete_sequence=px.colors.sequential.Blues
)
fig.show()

### 5. Area vs Price Scatter

In [6]:
fig = px.scatter(
    df, x="Gr Liv Area", y="SalePrice",
    color="Overall Qual",
    title="Living Area vs Sale Price (colored by Quality)",
    labels={
        "Gr Liv Area": "Above-Ground Living Area (sqft)",
        "SalePrice": "Sale Price (USD)"
    },
    opacity=0.6,
    color_continuous_scale="RdYlGn"
)
fig.show()

### 6. Correlation Heatmap (Top 15 features)

In [7]:
numeric_df = df.select_dtypes(include=[np.number])
correlations = numeric_df.corr()["SalePrice"].drop("SalePrice").abs()
top_features = correlations.nlargest(15).index.tolist()

corr_matrix = numeric_df[top_features + ["SalePrice"]].corr()

fig = px.imshow(
    corr_matrix,
    title="Correlation Heatmap — Top 15 Features vs SalePrice",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    text_auto=".2f"
)
fig.update_layout(width=800, height=700)
fig.show()

### 7. House Age vs Price

In [8]:
fig = px.scatter(
    df, x="house_age", y="SalePrice",
    trendline="lowess",
    title="House Age vs Sale Price",
    labels={"house_age": "House Age (years)", "SalePrice": "Sale Price"},
    opacity=0.4,
    color_discrete_sequence=["#2563eb"]
)
fig.show()

## Key EDA Takeaways

1. **Overall Quality** is the strongest single predictor (correlation ~0.80)
2. **Above-ground living area** is the second strongest (~0.71)
3. **Neighborhood** explains significant price variance, some areas 2–3× others
4. **House age** shows a negative trend, but recently remodeled homes recover value
5. **Price distribution** is right skewed that I will going to log-transform for modeling
6. Two extreme outliers removed (GrLivArea > 4000 sqft)